# Stage A - Training the Convolutional VAE
Train the healer network to reconstruct clean images from noisy ImageNet-100 inputs.

In [ ]:
from pathlib import Path
import yaml
import torch
import matplotlib.pyplot as plt
from torchinfo import summary

from src.dataset import get_dataloaders
from src.conv_vae import ConvVAE, vae_loss
from src.train_vae import train_vae
from src.evaluate import compute_psnr, compute_ssim

In [ ]:
with open('../configs/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

device = config['training']['device'] if torch.cuda.is_available() else 'cpu'
model = ConvVAE(latent_dim=config['vae']['latent_dim']).to(device)
model

In [ ]:
summary(model, input_size=(1, 3, 224, 224), depth=3)

In [ ]:
def loss_fn(recon, target, mu, logvar, beta):
    total, parts = vae_loss(recon, target, mu, logvar, beta=beta)
    return total, parts

loss_fn

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    root=config['dataset']['root'],
    image_size=config['dataset']['image_size'],
    batch_size=config['vae']['batch_size'],
    train_split=config['dataset']['train_split'],
    num_workers=config['training']['num_workers'],
    noisy=True,
    noise_type='gaussian',
    noise_params={'std': config['noise']['gaussian_std']}
)

history = train_vae(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=config['vae']['epochs'],
    learning_rate=config['vae']['learning_rate'],
    beta=config['vae']['beta'],
    save_path='../models/conv_vae_best.pth',
    patience=config['training']['early_stopping_patience']
)

In [ ]:
epochs = range(1, len(history['train_total']) + 1)
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(epochs, history['train_recon'], label='Train')
plt.plot(epochs, history['val_recon'], label='Val')
plt.title('Recon Loss')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs, history['train_kl'], label='Train')
plt.plot(epochs, history['val_kl'], label='Val')
plt.title('KL Loss')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs, history['train_total'], label='Train')
plt.plot(epochs, history['val_total'], label='Val')
plt.title('Total Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(torch.load('../models/conv_vae_best.pth', map_location=device))
model.eval()

noisy, clean, _ = next(iter(test_loader))
noisy = noisy.to(device)
clean = clean.to(device)
with torch.no_grad():
    recon, _, _ = model(noisy)

n = min(6, noisy.size(0))
fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
for i in range(n):
    axes[i, 0].imshow(noisy[i].detach().cpu().permute(1, 2, 0).numpy(), vmin=-2, vmax=2)
    axes[i, 0].set_title('Noisy')
    axes[i, 0].axis('off')
    axes[i, 1].imshow(recon[i].detach().cpu().permute(1, 2, 0).numpy())
    axes[i, 1].set_title('Reconstructed')
    axes[i, 1].axis('off')
    axes[i, 2].imshow(clean[i].detach().cpu().permute(1, 2, 0).numpy(), vmin=-2, vmax=2)
    axes[i, 2].set_title('Original')
    axes[i, 2].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
psnr_values = [compute_psnr(clean[i], recon[i]) for i in range(min(16, clean.size(0)))]
ssim_values = [compute_ssim(clean[i], recon[i]) for i in range(min(16, clean.size(0)))]
print('Avg PSNR:', sum(psnr_values) / len(psnr_values))
print('Avg SSIM:', sum(ssim_values) / len(ssim_values))

## Summary
The ConvVAE healer is trained, tracked with reconstruction and KL losses, and validated through qualitative reconstructions plus PSNR/SSIM metrics.